In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
  !pip install unsloth  # Do this in local & cloud setups
else:
  import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
  xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
  !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
  !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastVisionModel

model_path = '/content/drive/MyDrive/VisionModel/Llama-3.2-Vision'

dtype = None
max_seq_length = 4096

model, tokenizer = FastVisionModel.from_pretrained(
  model_name = model_path,
  load_in_4bit = True,
  dtype = dtype,
  max_seq_length = max_seq_length,
)

FastVisionModel.for_inference(model)

In [ ]:
instruction = """
### SYSTEM ROLE
    You are a Senior Cyber Threat Intelligence (CTI) Analyst and Knowledge Graph Engineer.
    Your task is to extract structured **Knowledge Graph Triples** from the provided CTI report text (or OCR output).
### OBJECTIVE
    Convert unstructured text into a list of triples in the format: `(Subject, Relation, Object)`.
    The output must be strictly JSON format.

    ### 1. ENTITY TYPES (Subject/Object)
    Classify entities into these specific types. Do not invent new types.
    - **Threat_Actor**: (e.g., APT29, LockBit, Lapsus$)
    - **Malware**: (e.g., Cobalt Strike, Mimikatz, WannaCry)
    - **Infrastructure**: (e.g., 192.168.1.5, bad-domain.com, C2 Server)
    - **Vulnerability**: (e.g., CVE-2023-1234, Log4Shell)
    - **Tool**: (e.g., n8n, PowerShell, curl)
    - **Indicator**: (e.g., "uid=1000(node)", specific file hash)
    - **Victim**: (e.g., TechCorp, Finance Sector, Database)
    - **Location**: (e.g., /var/www/html, Output Panel)

    ### 2. ALLOWED RELATIONS (Predicates)
    Use ONLY the following relationships. Do not use generic verbs like "is" or "has".
    - `uses` (e.g., Actor uses Malware)
    - `targets` (e.g., Malware targets Victim)
    - `communicates_with` (e.g., Malware communicates_with Infrastructure)
    - `exploits` (e.g., Tool exploits Vulnerability)
    - `drops` (e.g., Malware drops File)
    - `indicates` (e.g., String indicates Command Injection)
    - `located_at` (e.g., File located_at Path)
    - `variant_of` (e.g., LockBit 3.0 variant_of LockBit)
    - `executes` (e.g., LNK file executes BAT script)

    ### 3. STRICT RULES (Must Follow)
    1. **Atomicity:** Entities must be atomic.
      - BAD: "malicious C2 server at 192.168.1.1"
      - GOOD: ("192.168.1.1", "is_type", "C2 Server")
    2. **Standardization:**
      - Normalize IPs and Domains (remove brackets like [.]);
      - Use canonical names (e.g., use "LockBit" instead of "LockBit Ransomware" unless version differs).
    3. **No Hallucination:** Only extract relationships explicitly stated or visually confirmed in the text.
    4. **Directionality:** Always follow: Active Entity -> Passive Entity.
      - Correct: `Attacker` -> `targets` -> `Victim`
      - Incorrect: `Victim` -> `targeted_by` -> `Attacker`
    5. **Technical Precision:**
      - If text says `uid=1000(node)`, extract it as an `Indicator` entity.
      - Relationship: ("uid=1000(node)", "indicates", "Command Injection")

    ### 4. CHAIN OF THOUGHT PROCESS (Internal Reasoning)
    Before generating the final JSON, perform the following steps internally:
    1.  **Identify Entities:** Scan the text for potential entities and assign them one of the allowed Types. (e.g., "Found '10.0.0.5', type: Infrastructure").
    2.  **Determine Relations:** For each pair of entities, check if a relationship exists based on the Allowed Relations list. Verify the direction (Source -> Target).
    3.  **Refine & Filter:**
        - Check against Strict Rules (Atomicity, Standardization).
        - Remove any inferred relationships not explicitly in the text.
        - Ensure predicates are from the allowed list only.

    ### 5. OUTPUT FORMAT (JSON)
    Return a JSON object with a key "triples".


    ### 6. Expected outputs
    [
  {
    "text": "APT29, a threat group attributed to Russia's SVR, targeted several diplomatic entities using the Cobalt Strike beacon.",
    "triples": [
      {"subject": "APT29", "relation": "attributed-to", "object": "Russia"},
      {"subject": "APT29", "relation": "targets", "object": "Diplomatic Entities"},
      {"subject": "APT29", "relation": "uses", "object": "Cobalt Strike"}
    ]
  },
  {
    "text": "The BlackCat ransomware exploits the CVE-2021-44228 vulnerability to gain initial access and then encrypts victim files.",
    "triples": [
      {"subject": "BlackCat", "relation": "is-type-of", "object": "Ransomware"},
      {"subject": "BlackCat", "relation": "exploits", "object": "CVE-2021-44228"},
      {"subject": "BlackCat", "relation": "performs", "object": "File Encryption"}
    ]
  },
  {
    "text": "Kimsuky group uses spear-phishing emails containing malicious attachments to distribute the AppleSeed backdoor.",
    "triples": [
      {"subject": "Kimsuky", "relation": "uses-technique", "object": "Spear-Phishing"},
      {"subject": "Kimsuky", "relation": "distributes", "object": "AppleSeed Backdoor"},
      {"subject": "Spear-Phishing", "relation": "uses", "object": "Malicious Attachments"}
    ]
  }
]
"""

content = """
A maximum-severity security flaw in a WordPress plugin called Modular DS has come under active exploitation in the wild, according to Patchstack.

The vulnerability, tracked as CVE-2026-23550 (CVSS score: 10.0), has been described as a case of unauthenticated privilege escalation impacting all versions of the plugin prior to and including 2.5.1. It has been patched in version 2.5.2. The plugin has more than 40,000 active installs.

"In versions 2.5.1 and below, the plugin is vulnerable to privilege escalation, due to a combination of factors including direct route selection, bypassing of authentication mechanisms, and auto-login as admin," Patchstack said.

The problem is rooted in its routing mechanism, which is designed to put certain sensitive routes behind an authentication barrier. The plugin exposes its routes under the "/api/modular-connector/" prefix.
However, it has been found that this security layer can be bypassed every time the "direct request" mode is enabled by supplying an "origin" parameter set to "mo" and a "type" parameter set to any value (e.g., "origin=mo&type=xxx"). This causes the request to be treated as a Modular direct request.

"Therefore, as soon as the site has already been connected to Modular (tokens present/renewable), anyone can pass the auth middleware: there is no cryptographic link between the incoming request and Modular itself," Patchstack explained.

"This exposes several routes, including /login/, /server-information/, /manager/, and /backup/, which allow various actions to be performed, ranging from remote login to obtaining sensitive system or user data."

As a result of this loophole, an unauthenticated attacker can exploit the "/login/{modular_request}" route to get administrator access, resulting in privilege escalation. This could then pave the way for a full site compromise, permitting an attacker to introduce malicious changes, stage malware, or redirect users to scams.

According to details shared by the WordPress security company, attacks exploiting the flaw are said to have been first detected on January 13, 2026, at around 2 a.m. UTC, with HTTP GET calls to the endpoint "/api/modular-connector/login/" followed by attempts to create an admin user.

The attacks have originated from the following IP addresses -

    45.11.89[.]19
    185.196.0[.]11

In light of active exploitation of CVE-2026-23550, users of the plugin are advised to update to a patched version as soon as possible.
"This vulnerability highlights how dangerous implicit trust in internal request paths can be when exposed to the public internet," Patchstack said.

"In this case, the issue was not caused by a single bug, but by several design choices combined together: URL-based route matching, a permissive 'direct request' mode, authentication based only on the site connection state, and a login flow that automatically falls back to an administrator account."

Modular DS is also recommending users to review their sites for signs of compromise, such as unexpected admin users or suspicious requests from automated scanners, and, if found, perform the steps below -

    Regenerate WordPress salts to invalidate all existing sessions
    Regenerate OAuth credentials
    Scan the site for malicious plugins, files, or code

"The vulnerability was located in a custom routing layer extending Laravel’s route matching functionality," the maintainers of the plugin said. "The route matching logic was overly permissive, allowing crafted requests to match protected endpoints without proper authentication validation."
"""

message = [
    {
      "role": "system",
      "content": [
          {
              "type": "text",
              "text": instruction
          },
      ]
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": content
            },
        ]
    }
]

In [ ]:
inputs = tokenizer.apply_chat_template(
    message,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

In [ ]:
outputs = model.generate(
  input_ids = inputs,
  max_new_tokens = 1024,
  temperature=0.2,
  use_cache=True,
)